# H17: Perivascular Cell Phenotyping with Full RESTORE Normalization

**Goal:** Cell-type composition in perivascular neighborhoods of white pulp vessels, compared by rs3184504 genotype.

**Key improvements over H15/H16:**
- RESTORE normalization covers 21/22 markers (49 pairs) — no un-normalized markers dominating PCA
- Direct scored gating (fold-over-background) — no clustering topology dependency
- Memory/proliferating as annotations, not separate cell types — preserves CD4/CD8 identity
- Distance-ring vessel neighborhoods using pre-computed QuPath signed distances

In [ ]:
# ── Cell 1: Imports & Constants ──

import os
os.environ["NUMBA_NUM_THREADS"] = "64"
os.environ["OMP_NUM_THREADS"] = "64"
os.environ["OPENBLAS_NUM_THREADS"] = "64"

import warnings
warnings.filterwarnings('ignore', category=FutureWarning)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.mixture import GaussianMixture
from pathlib import Path

from data_utils import (
    extract_sample_id, GENOTYPE_MAP, CODEX_SAMPLES, EXCLUDE_SAMPLES,
    GENO_ORDER, GENO_PALETTE, assign_region_by_distance,
    full_stats_table, save_figure, save_table,
    setup_style, PROJECT, DATA_CSV,
)

setup_style()

# ---------------------------------------------------------------------------
# Marker definitions (22 total)
# ---------------------------------------------------------------------------
CELLS_CSV = PROJECT / "Measurements" / "Cells.csv"

COMMON_MARKERS_EXACT = [
    "CD45RO", "Ki67", "CD20", "CD4", "CD44", "CD31", "CD11c", "CD34",
    "CD107a", "CD163", "HLA-DR", "CD68", "CD8", "CD21", "Vimentin",
    "CD3e", "CD45", "Podoplanin",
]

HARMONIZE = [
    ("FOXP3",      "Cell: FOXP3: Mean",       "Cell: FoxP3: Mean"),
    ("SMA",        "Cell: SMA: Mean",          "Cell: SMActin: Mean"),
    ("CollagenIV", "Cell: Collagen IV: Mean",  "Cell: CollagenIV: Mean"),
    ("Ecadherin",  "Cell: E-cadherin: Mean",   "Cell: ECAD: Mean"),
]

ALL_MARKERS = COMMON_MARKERS_EXACT + [h[0] for h in HARMONIZE]  # 22

MARKER_COLS_EXACT = [f"Cell: {m}: Mean" for m in COMMON_MARKERS_EXACT]
HARMONIZE_COLS = [col for _, pc, cx in HARMONIZE for col in (pc, cx)]

# ---------------------------------------------------------------------------
# Column definitions
# ---------------------------------------------------------------------------
X_COL, Y_COL = "Centroid X µm", "Centroid Y µm"
DIST_COLS = [
    "Signed distance to annotation Follicle µm",
    "Signed distance to annotation PALS µm",
    "Signed distance to annotation RedPulp µm",
    "Signed distance to annotation SmallVessel µm",
    "Signed distance to annotation LargeVessel µm",
    "Signed distance to annotation Trabeculae µm",
]
META_COLS = ["Image", "Object ID", X_COL, Y_COL, "Cell: Area µm^2"]
MORPH_COLS = [
    "Nucleus: Area µm^2", "Cell: Area µm^2",
    "Nucleus/Cell area ratio", "Nucleus: Circularity", "Nucleus: Solidity",
]
DAPI_COL = "Cell: DAPI: Mean"

USE_COLS = list(dict.fromkeys(
    META_COLS + MORPH_COLS + [DAPI_COL] + MARKER_COLS_EXACT + HARMONIZE_COLS + DIST_COLS
))

# ---------------------------------------------------------------------------
# Crop parameters
# ---------------------------------------------------------------------------
CROP_W, CROP_H = 5000, 3000
CROP_THRESHOLD_X, CROP_THRESHOLD_Y = 5000, 3000
CROP_OVERRIDES = {"HDL011": (2000, 2000)}

# ---------------------------------------------------------------------------
# QC thresholds
# ---------------------------------------------------------------------------
NUC_AREA_MIN, NUC_AREA_MAX = 5, 200
CELL_AREA_MIN, CELL_AREA_MAX = 10, 500
NC_RATIO_MIN, NC_RATIO_MAX = 0.1, 1.0
NUC_CIRC_MIN = 0.15
NUC_SOL_MIN = 0.5
MAX_ZERO_MARKERS = 18
ZERO_THRESHOLD = 1.0

# ---------------------------------------------------------------------------
# RESTORE constants
# ---------------------------------------------------------------------------
RESTORE_SIGMA_WEIGHT = 3
RESTORE_NEG_QUANTILE = 0.75

# ---------------------------------------------------------------------------
# Dev mode
# ---------------------------------------------------------------------------
DEV_MODE = True
DEV_IMAGES_SAMPLES = {"HDL055", "HDL052", "1901HBMP004"}

# ---------------------------------------------------------------------------
# Vessel neighborhood constants
# ---------------------------------------------------------------------------
PERI_RADIUS = 25    # µm — perivascular zone
PROX_RADIUS = 75    # µm — proximal zone

# ---------------------------------------------------------------------------
# Genotype helpers
# ---------------------------------------------------------------------------
DOSAGE_MAP = {"C/C": 0, "C/T": 1, "T/T": 2}

def _annotate_dosage(ax, stats_df):
    """Add Spearman dosage rho/p annotation to top of axis."""
    row = stats_df[stats_df["Test"] == "Spearman dosage"]
    if row.empty:
        return
    p = row["p"].iloc[0]
    es = row["Effect_Size"].iloc[0]
    if isinstance(es, str) and es.startswith("rho="):
        rho_str = es
    else:
        rho_str = f"rho={row['Statistic'].iloc[0]:.2f}"
    color = "#CC3311" if p < 0.05 else ("#EE7733" if p < 0.1 else "grey")
    ax.annotate(f"{rho_str}, p={p:.3f}", xy=(0.5, 1.0), xycoords="axes fraction",
                ha="center", va="bottom", fontsize=7, color=color)

print(f"All markers: {len(ALL_MARKERS)}")
print(f"Columns to load: {len(USE_COLS)}")
print(f"DEV_MODE: {DEV_MODE} → samples: {DEV_IMAGES_SAMPLES if DEV_MODE else 'all'}")

In [ ]:
# ── Cell 2: Expanded RESTORE Pairs (49 total, 21/22 markers) ──

RESTORE_PAIRS = [
    # ── Existing (H16) — 19 pairs, 11 targets ──
    ("CD20",  "CD3e"),    ("CD3e",  "CD20"),
    ("CD4",   "CD8"),     ("CD8",   "CD4"),
    ("CD20",  "CD68"),    ("CD68",  "CD20"),
    ("CD3e",  "CD68"),    ("CD68",  "CD3e"),
    ("Vimentin","CD20"),  ("Vimentin","CD3e"),
    ("CD31",  "CD20"),    ("CD31",  "CD3e"),
    ("SMA",   "CD20"),    ("SMA",   "CD3e"),
    ("CD163", "CD20"),
    ("CD11c", "CD20"),    ("CD11c", "CD3e"),
    ("CD21",  "CD3e"),    ("CD21",  "CD68"),
    # ── New (H17) — 30 pairs, 10 new targets ──
    # CD45RO (memory T) → negative in structural + naive B
    ("CD45RO","Vimentin"),("CD45RO","CD31"),("CD45RO","SMA"),("CD45RO","CD20"),
    # FOXP3 (Treg) → negative in all non-T
    ("FOXP3", "CD20"),("FOXP3", "CD68"),("FOXP3", "Vimentin"),("FOXP3", "CD31"),
    # Ki67 (proliferation) → negative in quiescent structural
    ("Ki67",  "SMA"),("Ki67",  "Vimentin"),("Ki67",  "CD31"),
    # CD34 (endothelial/progenitor) → negative in mature immune
    ("CD34",  "CD20"),("CD34",  "CD3e"),("CD34",  "CD68"),
    # Podoplanin (FRC/lymphatic) → negative in immune
    ("Podoplanin","CD20"),("Podoplanin","CD3e"),("Podoplanin","CD68"),
    # HLA-DR (MHC-II) → negative in structural (normal spleen)
    ("HLA-DR","SMA"),("HLA-DR","Vimentin"),("HLA-DR","CD31"),
    # CD107a (degranulation) → negative in structural
    ("CD107a","SMA"),("CD107a","Vimentin"),("CD107a","CD31"),
    # CollagenIV (ECM) → negative in immune
    ("CollagenIV","CD20"),("CollagenIV","CD3e"),("CollagenIV","CD68"),
    # E-cadherin → negative in mesenchymal/T
    ("Ecadherin","Vimentin"),("Ecadherin","CD3e"),("Ecadherin","CD31"),
    # CD45 (pan-leukocyte) → negative in structural
    ("CD45",  "Vimentin"),("CD45",  "SMA"),("CD45",  "CD31"),
]
# CD44 has NO clean exclusive partner — handled by percentile normalization

# Validate
targets_covered = sorted(set(p[0] for p in RESTORE_PAIRS))
uncovered = [m for m in ALL_MARKERS if m not in targets_covered]
print(f"RESTORE pairs: {len(RESTORE_PAIRS)}")
print(f"Targets covered: {len(targets_covered)}/22 — {targets_covered}")
print(f"Uncovered (percentile fallback): {uncovered}")

In [ ]:
# ── Cell 3: Data Loading & Crop Filter ──

# Step 1: Compute crop windows from AnnotationsFinal.csv
annot = pd.read_csv(DATA_CSV, usecols=["Image", X_COL, Y_COL])
annot["Sample"] = annot["Image"].apply(extract_sample_id)
annot = annot[~annot["Sample"].isin(EXCLUDE_SAMPLES)]

half_w, half_h = CROP_W / 2, CROP_H / 2
crop_windows = {}
img_to_sample = {}

for sample in sorted(annot["Sample"].unique()):
    sdf = annot[annot["Sample"] == sample]
    for img in sdf["Image"].unique():
        img_to_sample[img] = sample
    coords = sdf[[X_COL, Y_COL]].values
    x_ext = coords[:, 0].ptp()
    y_ext = coords[:, 1].ptp()

    if x_ext <= CROP_THRESHOLD_X and y_ext <= CROP_THRESHOLD_Y:
        crop_windows[sample] = None
        continue

    cx = np.median(coords[:, 0])
    cy = np.median(coords[:, 1])

    if sample in CROP_OVERRIDES:
        dx, dy = CROP_OVERRIDES[sample]
        cx += dx
        cy += dy

    cx = np.clip(cx, coords[:, 0].min() + half_w, coords[:, 0].max() - half_w)
    cy = np.clip(cy, coords[:, 1].min() + half_h, coords[:, 1].max() - half_h)
    crop_windows[sample] = (cx, cy)

del annot
analysis_images = set(img_to_sample.keys())

print(f"Crop windows for {len(crop_windows)} samples:")
for s in sorted(crop_windows):
    w = crop_windows[s]
    tag = "no crop" if w is None else f"cx={w[0]:.0f}, cy={w[1]:.0f}"
    ov = " (override)" if s in CROP_OVERRIDES else ""
    print(f"  {s:16s} {tag}{ov}")

# Step 2: Load Cells.csv
cells = pd.read_csv(CELLS_CSV, usecols=USE_COLS, engine='pyarrow')
n_total = len(cells)

# Step 3: Filter to analysis images, add Sample
cells = cells[cells["Image"].isin(analysis_images)].copy()
cells["Sample"] = cells["Image"].map(img_to_sample)

# Dev mode filter
if DEV_MODE:
    cells = cells[cells["Sample"].isin(DEV_IMAGES_SAMPLES)].copy()
    print(f"\nDEV_MODE: filtered to {DEV_IMAGES_SAMPLES}")

# Step 4: Apply crop windows
keep = np.ones(len(cells), dtype=bool)
for sample, window in crop_windows.items():
    if window is None:
        continue
    mask = cells["Sample"].values == sample
    if not mask.any():
        continue
    cx, cy = window
    keep[mask] &= (
        (np.abs(cells.loc[mask, X_COL].values - cx) <= half_w) &
        (np.abs(cells.loc[mask, Y_COL].values - cy) <= half_h))

n_pre_crop = len(cells)
cells = cells[keep].reset_index(drop=True)
n_post_crop = len(cells)

print(f"\nLoaded {n_total:,} total cells")
print(f"Analysis images: {n_pre_crop:,}")
print(f"After crop: {n_post_crop:,} (removed {n_pre_crop - n_post_crop:,})")
print(f"\nPer-sample cell counts:")
for s in sorted(cells["Sample"].unique()):
    print(f"  {s:16s} {(cells['Sample'] == s).sum():>8,}")

In [ ]:
# ── Cell 4: Morphological QC & Marker Harmonization ──

# QC filters
morph_ok = (
    (cells["Nucleus: Area µm^2"] >= NUC_AREA_MIN) &
    (cells["Nucleus: Area µm^2"] <= NUC_AREA_MAX) &
    (cells["Cell: Area µm^2"] >= CELL_AREA_MIN) &
    (cells["Cell: Area µm^2"] <= CELL_AREA_MAX) &
    (cells["Nucleus/Cell area ratio"] >= NC_RATIO_MIN) &
    (cells["Nucleus/Cell area ratio"] <= NC_RATIO_MAX) &
    (cells["Nucleus: Circularity"] >= NUC_CIRC_MIN) &
    (cells["Nucleus: Solidity"] >= NUC_SOL_MIN)
)
n_morph_fail = (~morph_ok).sum()
cells = cells[morph_ok].reset_index(drop=True)
n_post_morph = len(cells)

# DAPI floor (1st percentile per image)
dapi_fail = np.zeros(len(cells), dtype=bool)
for img in cells["Image"].unique():
    img_mask = cells["Image"].values == img
    dapi_vals = cells.loc[img_mask, DAPI_COL].values
    floor = np.percentile(dapi_vals, 1)
    dapi_fail[img_mask] = dapi_vals < floor
n_dapi_fail = dapi_fail.sum()
cells = cells[~dapi_fail].reset_index(drop=True)
n_post_dapi = len(cells)

# Multi-marker zero filter
raw_vals = cells[MARKER_COLS_EXACT].values
n_near_zero = (raw_vals < ZERO_THRESHOLD).sum(axis=1)
zero_thresh = int(len(MARKER_COLS_EXACT) * MAX_ZERO_MARKERS / 22)
multi_zero = n_near_zero > zero_thresh
n_zero_fail = multi_zero.sum()
cells = cells[~multi_zero].reset_index(drop=True)
n_post_zero = len(cells)

# Harmonize 4 marker pairs
for canonical, pc_col, cx_col in HARMONIZE:
    canon_col = f"Cell: {canonical}: Mean"
    cells[canon_col] = cells[pc_col].combine_first(cells[cx_col])
    to_drop = [c for c in (pc_col, cx_col) if c != canon_col]
    cells.drop(columns=to_drop, inplace=True, errors='ignore')

# Final marker column names
all_marker_cols = [f"Cell: {m}: Mean" for m in ALL_MARKERS]

# Check NaN and drop
nan_mask = cells[all_marker_cols].isna().any(axis=1)
n_nan = nan_mask.sum()
cells = cells[~nan_mask].reset_index(drop=True)

# Add metadata
cells["Genotype"] = cells["Sample"].map(GENOTYPE_MAP)
cells["Platform"] = cells["Sample"].apply(
    lambda s: "CODEX" if s in CODEX_SAMPLES else "Phenocycler")
cells["Dosage"] = cells["Genotype"].map(DOSAGE_MAP)

# Assign region by signed distance
cells["Region"] = assign_region_by_distance(
    cells["Signed distance to annotation Follicle µm"].values,
    cells["Signed distance to annotation PALS µm"].values,
    cells["Signed distance to annotation RedPulp µm"].values,
)

# QC summary
print("=" * 60)
print("QC Waterfall")
print("=" * 60)
print(f"  After crop:           {n_post_crop:>10,}")
print(f"  After morph QC:       {n_post_morph:>10,}  (removed {n_morph_fail:,})")
print(f"  After DAPI floor:     {n_post_dapi:>10,}  (removed {n_dapi_fail:,})")
print(f"  After zero filter:    {n_post_zero:>10,}  (removed {n_zero_fail:,})")
print(f"  After NaN drop:       {len(cells):>10,}  (removed {n_nan:,})")
print(f"\n  Final: {len(cells):,} cells")
print("=" * 60)
print(f"\nGenotype: {dict(cells['Genotype'].value_counts().sort_index())}")
print(f"Platform: {cells['Platform'].value_counts().to_dict()}")
print(f"Region:   {cells['Region'].value_counts().to_dict()}")

In [ ]:
# ── Cell 5: RESTORE Normalization ──

RESTORE_MIN_RANGE = 1.0   # skip pairs where threshold-baseline < this (degenerate GMM)
RESTORE_MAX_FOLD = 20.0   # clip fold values — anything above this is equally "positive"

def restore_normalize(cells, marker_cols, pairs, sigma_weight=3, neg_quantile=0.75,
                      min_range=RESTORE_MIN_RANGE, max_fold=RESTORE_MAX_FOLD):
    """RESTORE normalization: per-image, per-marker background via GMM on
    mutually exclusive marker pairs.

    Returns: (normalized_cells DataFrame, diagnostics DataFrame)
    """
    all_markers = [col.replace("Cell: ", "").replace(": Mean", "") for col in marker_cols]
    m_to_idx = {m: i for i, m in enumerate(all_markers)}

    raw = cells[marker_cols].values.astype(np.float64)
    images = cells["Image"].values
    unique_images = cells["Image"].unique()
    normalized = raw.copy()

    diag_rows = []

    for img in unique_images:
        img_mask = images == img
        img_raw = raw[img_mask]

        marker_thresholds = {m: [] for m in all_markers}
        marker_baselines = {m: [] for m in all_markers}

        for target, partner in pairs:
            if target not in m_to_idx or partner not in m_to_idx:
                continue
            t_idx = m_to_idx[target]
            p_idx = m_to_idx[partner]

            partner_vals = img_raw[:, p_idx]
            partner_thresh = np.percentile(partner_vals, neg_quantile * 100)
            neg_mask = partner_vals > partner_thresh
            n_neg = neg_mask.sum()

            if n_neg < 50:
                continue

            try:
                fit_data = np.column_stack([
                    img_raw[neg_mask, t_idx],
                    img_raw[neg_mask, p_idx]
                ])
                gmm = GaussianMixture(
                    n_components=2, n_init=10, covariance_type='full',
                    random_state=42
                )
                gmm.fit(fit_data)

                neg_comp = np.argmax([
                    np.diagonal(cov)[1] for cov in gmm.covariances_
                ])

                mu = gmm.means_[neg_comp, 0]
                sigma = np.sqrt(np.diagonal(gmm.covariances_[neg_comp])[0])
                thresh = mu + sigma_weight * sigma
                pair_range = thresh - mu

                tier = "OK"
                if pair_range < min_range:
                    tier = "DEGENERATE"  # skip this pair
                else:
                    marker_thresholds[target].append(thresh)
                    marker_baselines[target].append(mu)

                diag_rows.append({
                    "Image": img, "Marker": target, "Partner": partner,
                    "N_neg_cells": n_neg, "GMM_mu": mu, "GMM_sigma": sigma,
                    "Threshold": thresh, "Range": pair_range, "Tier": tier,
                })
            except Exception:
                continue

        # Apply normalization for this image
        for m_idx, marker in enumerate(all_markers):
            threshs = marker_thresholds[marker]
            bases = marker_baselines[marker]
            if not threshs:
                diag_rows.append({
                    "Image": img, "Marker": marker, "Partner": "NONE",
                    "N_neg_cells": 0, "GMM_mu": np.nan, "GMM_sigma": np.nan,
                    "Threshold": np.nan, "Range": np.nan, "Tier": "NO_PAIRS",
                })
                continue

            baseline = np.median(bases)
            threshold = np.median(threshs)

            if threshold <= baseline:
                continue

            vals = img_raw[:, m_idx]
            normed = (vals - baseline) / (threshold - baseline)
            normed = np.clip(normed, 0, max_fold)
            normalized[img_mask, m_idx] = normed

    diagnostics = pd.DataFrame(diag_rows)
    result = cells.copy()
    for i, col in enumerate(marker_cols):
        result[col] = normalized[:, i]
    return result, diagnostics

# Run RESTORE
cells, restore_diag = restore_normalize(
    cells, all_marker_cols, RESTORE_PAIRS,
    sigma_weight=RESTORE_SIGMA_WEIGHT,
    neg_quantile=RESTORE_NEG_QUANTILE,
)

# FOXP3 biological restriction: zero for cells NOT (CD3e+ AND CD4+)
cd3e_col = "Cell: CD3e: Mean"
cd4_col = "Cell: CD4: Mean"
foxp3_col = "Cell: FOXP3: Mean"

cd3e_pos = cells[cd3e_col] > 0
cd4_pos = cells[cd4_col] > 0
not_cd3cd4 = ~(cd3e_pos & cd4_pos)
n_zeroed = not_cd3cd4.sum()
cells.loc[not_cd3cd4, foxp3_col] = 0.0
print(f"FOXP3 zeroed for {n_zeroed:,} / {len(cells):,} cells "
      f"({100*n_zeroed/len(cells):.1f}%) that are not CD3e+CD4+")

# CD44 percentile normalization (no RESTORE partner)
cd44_col = "Cell: CD44: Mean"
for img in cells["Image"].unique():
    img_mask = cells["Image"].values == img
    vals = cells.loc[img_mask, cd44_col].values
    p1 = np.percentile(vals, 1)
    p99 = np.percentile(vals, 99)
    if p99 > p1:
        normed = (vals - p1) / (p99 - p1)
        cells.loc[img_mask, cd44_col] = np.clip(normed, 0, 1)
print(f"CD44 percentile-normalized per image")

# Summary
diag_valid = restore_diag[restore_diag["Tier"] == "OK"]
diag_degen = restore_diag[restore_diag["Tier"] == "DEGENERATE"]
diag_none = restore_diag[restore_diag["Tier"] == "NO_PAIRS"]
n_markers_with_pairs = diag_valid["Marker"].nunique()

print(f"\nRESTORE normalization complete:")
print(f"  {n_markers_with_pairs}/{len(ALL_MARKERS)} markers had valid pairs")
print(f"  {len(diag_valid)} OK pairs, {len(diag_degen)} degenerate (skipped), "
      f"{len(diag_none)} no-pairs")

if len(diag_degen):
    print(f"\n  Degenerate pairs (range < {RESTORE_MIN_RANGE}):")
    for _, r in diag_degen.iterrows():
        print(f"    {r['Image'][:20]:20s}  {r['Marker']:12s} ← {r['Partner']:8s}  "
              f"range={r['Range']:.4f}")

if len(diag_valid):
    summary = diag_valid.groupby("Marker").agg(
        n_pairs=("Partner", "count"),
        n_images=("Image", "nunique"),
        median_baseline=("GMM_mu", "median"),
        median_threshold=("Threshold", "median"),
    )
    print(f"\nPer-marker RESTORE summary:")
    for marker in ALL_MARKERS:
        if marker in summary.index:
            row = summary.loc[marker]
            print(f"  {marker:16s}  pairs={row['n_pairs']:3.0f}  "
                  f"baseline={row['median_baseline']:7.1f}  "
                  f"threshold={row['median_threshold']:7.1f}")
        else:
            print(f"  {marker:16s}  NO VALID PAIRS (percentile fallback)")

if len(diag_none):
    no_pair_markers = diag_none["Marker"].unique()
    no_pair_images = diag_none.groupby("Marker")["Image"].apply(list).to_dict()
    print(f"\n  Markers with no valid pairs in some images:")
    for m, imgs in no_pair_images.items():
        img_short = [i[:20] for i in imgs]
        print(f"    {m:16s} → {img_short}")

save_table(restore_diag, "H17_restore_diagnostics")

In [ ]:
# ── Cell 6: Direct Scored Gating ──

PHENOTYPE_RULES = {
    "B cell":        {"pos": ["CD20"],                "neg": ["CD3e"],              "neg_w": 0.5},
    "CD4 T cell":    {"pos": ["CD3e", "CD4"],         "neg": ["CD20", "CD8"],       "neg_w": 0.3},
    "CD8 T cell":    {"pos": ["CD3e", "CD8"],         "neg": ["CD20", "CD4"],       "neg_w": 0.3},
    "T-reg":         {"pos": ["CD3e", "CD4", "FOXP3"],"neg": ["CD20", "CD8"],       "neg_w": 0.3},
    "FDC":           {"pos": ["CD21"],                "neg": ["CD3e", "CD68"],      "neg_w": 0.3},
    "Macrophage":    {"pos": ["CD68"],                "neg": ["CD20", "CD3e"],      "neg_w": 0.3},
    "M2 Macrophage": {"pos": ["CD163", "CD68"],       "neg": ["CD20", "CD3e"],      "neg_w": 0.3},
    "DC":            {"pos": ["CD11c", "HLA-DR"],     "neg": ["CD20", "CD68"],      "neg_w": 0.3},
    "Endothelial":   {"pos": ["CD31"],                "neg": [],                    "neg_w": 0},
    "SMC":           {"pos": ["SMA"],                 "neg": [],                    "neg_w": 0},
    "Stromal":       {"pos": ["Vimentin"],            "neg": ["CD20","CD3e","CD68","CD31"], "neg_w": 0.2},
}

CELL_TYPE_ORDER = [
    "B cell", "CD4 T cell", "CD8 T cell", "T-reg",
    "FDC", "Macrophage", "M2 Macrophage", "DC",
    "Endothelial", "SMC", "Stromal", "Untyped",
]

ct_palette = dict(zip(CELL_TYPE_ORDER, [
    "#1f77b4", "#2ca02c", "#d62728", "#bcbd22",
    "#ff7f0e", "#9467bd", "#c5b0d5", "#8c564b",
    "#e377c2", "#f7b6d2", "#7f7f7f", "#cccccc",
]))

# Compute scores for all cell types
marker_col_map = {m: f"Cell: {m}: Mean" for m in ALL_MARKERS}
type_names = list(PHENOTYPE_RULES.keys())
n_types = len(type_names)
n_cells = len(cells)
scores = np.zeros((n_cells, n_types), dtype=np.float64)

for j, (ct, rule) in enumerate(PHENOTYPE_RULES.items()):
    pos_cols = [marker_col_map[m] for m in rule["pos"]]
    neg_cols = [marker_col_map[m] for m in rule["neg"]]

    pos_vals = cells[pos_cols].values
    pos_mean = pos_vals.mean(axis=1)

    if neg_cols:
        neg_vals = cells[neg_cols].values
        neg_mean = neg_vals.mean(axis=1)
        scores[:, j] = pos_mean - rule["neg_w"] * neg_mean
    else:
        scores[:, j] = pos_mean

# Assign: highest score wins if > 0
best_idx = np.argmax(scores, axis=1)
best_score = scores[np.arange(n_cells), best_idx]

# Second-best for confidence
sorted_scores = np.sort(scores, axis=1)
second_best = sorted_scores[:, -2]
confidence = np.where(best_score > 0,
                      (best_score - second_best) / np.maximum(best_score, 1e-10),
                      0.0)

cell_types = np.array([type_names[i] for i in best_idx], dtype=object)
cell_types[best_score <= 0] = "Untyped"

# Check that all positive markers are above 0 for assignment
for j, (ct, rule) in enumerate(PHENOTYPE_RULES.items()):
    pos_cols = [marker_col_map[m] for m in rule["pos"]]
    pos_vals = cells[pos_cols].values
    all_pos_above_zero = (pos_vals > 0).all(axis=1)
    # If assigned this type but not all positive markers are above 0, mark Untyped
    assigned = cell_types == ct
    cell_types[assigned & ~all_pos_above_zero] = "Untyped"
    confidence[assigned & ~all_pos_above_zero] = 0.0

cells["cell_type"] = cell_types
cells["ct_confidence"] = confidence

# Secondary annotations (do NOT overwrite cell_type)
t_cell_mask = cells["cell_type"].isin(["CD4 T cell", "CD8 T cell", "T-reg"])
cells["is_memory"] = False
cells.loc[t_cell_mask & (cells["Cell: CD45RO: Mean"] > 0), "is_memory"] = True

cells["is_proliferating"] = cells["Cell: Ki67: Mean"] > 0

# Summary
print("Cell type distribution:")
ct_counts = cells["cell_type"].value_counts()
for ct in CELL_TYPE_ORDER:
    if ct in ct_counts.index:
        n = ct_counts[ct]
        print(f"  {ct:18s} {n:>8,}  ({100*n/len(cells):5.1f}%)")

untyped_frac = (cells["cell_type"] == "Untyped").mean()
print(f"\nUntyped: {100*untyped_frac:.1f}% (target: <15%)")
print(f"Confidence: mean={cells['ct_confidence'].mean():.3f}, "
      f"median={cells['ct_confidence'].median():.3f}")
print(f"\nMemory T cells: {cells['is_memory'].sum():,} / {t_cell_mask.sum():,} T cells")
print(f"Proliferating: {cells['is_proliferating'].sum():,} / {len(cells):,} cells")

In [ ]:
# ── Cell 7: Validation — Marker Heatmap ──

# Phenotyping markers used in scoring
pheno_markers = sorted(set(
    m for rule in PHENOTYPE_RULES.values()
    for m in rule["pos"] + rule["neg"]
))

# Compute median fold-over-background per cell type per marker
typed = cells[cells["cell_type"] != "Untyped"]
ct_order = [ct for ct in CELL_TYPE_ORDER if ct != "Untyped" and ct in typed["cell_type"].unique()]

heatmap_data = []
for ct in ct_order:
    ct_cells = typed[typed["cell_type"] == ct]
    row = {}
    for m in pheno_markers:
        col = marker_col_map[m]
        row[m] = ct_cells[col].median()
    heatmap_data.append(row)

heatmap_df = pd.DataFrame(heatmap_data, index=ct_order)

fig, ax = plt.subplots(figsize=(12, 6))
sns.heatmap(heatmap_df, annot=True, fmt=".2f", cmap="YlOrRd", linewidths=0.5,
            ax=ax, vmin=0, vmax=heatmap_df.values.max() * 0.8)
ax.set_title("Median RESTORE fold-over-background by cell type")
ax.set_ylabel("Cell type")
ax.set_xlabel("Marker")
plt.tight_layout()
save_figure(fig, "H17_marker_heatmap")
plt.show()

# Flag cell types where defining markers are not highest
print("\nValidation check — defining markers should be among highest:")
for ct in ct_order:
    rule = PHENOTYPE_RULES.get(ct, {})
    pos = rule.get("pos", [])
    row = heatmap_df.loc[ct]
    top_markers = row.nlargest(len(pos)).index.tolist()
    all_in_top = all(m in top_markers for m in pos)
    status = "OK" if all_in_top else "WARNING"
    print(f"  {ct:18s}  defining={pos}  top-{len(pos)}={top_markers}  [{status}]")

In [ ]:
# ── Cell 8: Validation — Spatial Coherence ──

# Expected region for each cell type (signed distance < 0 = inside)
EXPECTED_REGIONS = {
    "B cell":       "Follicle",
    "CD4 T cell":   "PALS",
    "CD8 T cell":   "RedPulp",
    "T-reg":        "PALS",
    "FDC":          "Follicle",
    "Macrophage":   "RedPulp",
    "M2 Macrophage":"RedPulp",
    "DC":           "PALS",
    "Endothelial":  None,  # ubiquitous
    "SMC":          None,
    "Stromal":      None,
}

# % of each cell type inside each region
regions_check = ["Follicle", "PALS", "RedPulp"]
region_dist_cols = {
    "Follicle": "Signed distance to annotation Follicle µm",
    "PALS":     "Signed distance to annotation PALS µm",
    "RedPulp":  "Signed distance to annotation RedPulp µm",
}

coherence_rows = []
typed = cells[cells["cell_type"] != "Untyped"]
for ct in [c for c in CELL_TYPE_ORDER if c != "Untyped" and c in typed["cell_type"].unique()]:
    ct_cells = typed[typed["cell_type"] == ct]
    row = {"Cell Type": ct, "N": len(ct_cells)}
    for region, dist_col in region_dist_cols.items():
        inside = (ct_cells[dist_col] < 0).mean() * 100
        row[f"% in {region}"] = inside
    coherence_rows.append(row)

coherence_df = pd.DataFrame(coherence_rows)
print("Spatial coherence — % of cell type inside each region:")
print(coherence_df.to_string(index=False, float_format="%.1f"))

# Check thresholds
print("\nValidation:")
for ct, expected in EXPECTED_REGIONS.items():
    if expected is None or ct not in coherence_df["Cell Type"].values:
        continue
    pct = coherence_df.loc[coherence_df["Cell Type"] == ct, f"% in {expected}"].values[0]
    threshold = 50 if ct == "B cell" else 30
    status = "OK" if pct >= threshold else "LOW"
    print(f"  {ct:18s} in {expected:10s}: {pct:5.1f}% [{status}]")

# Spatial scatter — dev images
plot_cells = typed.copy()
fig, axes = plt.subplots(1, len(plot_cells["Sample"].unique()), figsize=(7 * len(plot_cells["Sample"].unique()), 6))
if not hasattr(axes, '__len__'):
    axes = [axes]

for ax, sample in zip(axes, sorted(plot_cells["Sample"].unique())):
    sc = plot_cells[plot_cells["Sample"] == sample]
    for ct in CELL_TYPE_ORDER:
        if ct == "Untyped":
            continue
        ct_mask = sc["cell_type"] == ct
        if ct_mask.sum() == 0:
            continue
        ax.scatter(sc.loc[ct_mask, X_COL], sc.loc[ct_mask, Y_COL],
                   c=ct_palette.get(ct, "#cccccc"), s=0.3, alpha=0.4, label=ct, rasterized=True)
    ax.set_title(f"{sample}\n({GENOTYPE_MAP.get(sample, '?')})", fontsize=10)
    ax.set_aspect("equal")
    ax.invert_yaxis()
    ax.set_xlabel("X (µm)")
    ax.set_ylabel("Y (µm)")

# Single legend
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", ncol=6, fontsize=7,
           bbox_to_anchor=(0.5, 1.05), markerscale=5)
fig.suptitle("Cell type spatial maps (crop windows)", y=1.08, fontsize=12)
plt.tight_layout()
save_figure(fig, "H17_spatial_celltype_maps")
plt.show()

In [ ]:
# ── Cell 9: Vessel Neighborhood Definition ──

dist_sv_col = "Signed distance to annotation SmallVessel µm"
dist_fol_col = "Signed distance to annotation Follicle µm"
dist_pals_col = "Signed distance to annotation PALS µm"

dist_sv = cells[dist_sv_col].values

# Distance rings from SmallVessel boundary
cells["vessel_zone"] = np.where(
    np.abs(dist_sv) <= PERI_RADIUS, "Perivascular",
    np.where(np.abs(dist_sv) <= PROX_RADIUS, "Proximal", "Distal"))

# White pulp restriction: inside Follicle OR PALS (signed distance < 0)
dist_fol = cells[dist_fol_col].values
dist_pals = cells[dist_pals_col].values
cells["in_white_pulp"] = (dist_fol < 0) | (dist_pals < 0)

# Sub-region: which WP compartment
cells["wp_compartment"] = np.where(
    dist_fol < 0, "Follicle",
    np.where(dist_pals < 0, "PALS", "Outside"))

# Summary
print("Vessel zone distribution (all cells):")
zone_counts = cells["vessel_zone"].value_counts()
for z in ["Perivascular", "Proximal", "Distal"]:
    if z in zone_counts.index:
        print(f"  {z:16s} {zone_counts[z]:>8,}  ({100*zone_counts[z]/len(cells):.1f}%)")

wp_cells = cells[cells["in_white_pulp"]]
print(f"\nWhite pulp cells: {len(wp_cells):,} / {len(cells):,} ({100*len(wp_cells)/len(cells):.1f}%)")
print(f"\nWhite pulp × vessel zone:")
wp_zone = wp_cells.groupby("vessel_zone").size()
for z in ["Perivascular", "Proximal", "Distal"]:
    if z in wp_zone.index:
        print(f"  {z:16s} {wp_zone[z]:>8,}")

print(f"\nWP compartment × vessel zone:")
cross = pd.crosstab(wp_cells["wp_compartment"], wp_cells["vessel_zone"])
print(cross.to_string())

print(f"\nPer-sample white pulp perivascular cell counts:")
for s in sorted(wp_cells["Sample"].unique()):
    n = ((wp_cells["Sample"] == s) & (wp_cells["vessel_zone"] == "Perivascular")).sum()
    print(f"  {s:16s} {n:>6,}")

In [ ]:
# ── Cell 10: Perivascular Cell Composition by Genotype ──

wp = cells[cells["in_white_pulp"]].copy()
typed_ct = [ct for ct in CELL_TYPE_ORDER if ct != "Untyped"]

# Per-image, per-zone cell-type proportions
prop_rows = []
for sample in sorted(wp["Sample"].unique()):
    geno = GENOTYPE_MAP.get(sample, None)
    if geno is None:
        continue
    for zone in ["Perivascular", "Proximal", "Distal"]:
        zone_cells = wp[(wp["Sample"] == sample) & (wp["vessel_zone"] == zone)]
        n_zone = len(zone_cells)
        if n_zone == 0:
            continue
        ct_counts = zone_cells["cell_type"].value_counts()
        row = {"Sample": sample, "Genotype": geno, "Zone": zone, "N_cells": n_zone}
        for ct in typed_ct:
            row[ct] = ct_counts.get(ct, 0) / n_zone
        prop_rows.append(row)

prop_df = pd.DataFrame(prop_rows)
prop_df["Genotype"] = pd.Categorical(prop_df["Genotype"], categories=GENO_ORDER, ordered=True)

# --- Stacked bar: perivascular proportions by genotype ---
peri = prop_df[prop_df["Zone"] == "Perivascular"].sort_values("Genotype")
fig, ax = plt.subplots(figsize=(10, 5))
bottom = np.zeros(len(peri))
x = np.arange(len(peri))
for ct in typed_ct:
    if ct not in peri.columns:
        continue
    vals = peri[ct].values
    ax.bar(x, vals, bottom=bottom, color=ct_palette.get(ct, "#cccccc"), label=ct, width=0.7)
    bottom += vals

labels = [f"{r['Sample']}\n({r['Genotype']})" for _, r in peri.iterrows()]
ax.set_xticks(x)
ax.set_xticklabels(labels, fontsize=8, rotation=45, ha="right")
ax.set_ylabel("Proportion")
ax.set_title("Perivascular cell composition — White Pulp")
ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=7)
plt.tight_layout()
save_figure(fig, "H17_perivascular_stacked_bar")
plt.show()

# --- Enrichment: perivascular / distal fraction ---
enrichment_rows = []
for sample in sorted(wp["Sample"].unique()):
    geno = GENOTYPE_MAP.get(sample)
    if geno is None:
        continue
    peri_row = prop_df[(prop_df["Sample"] == sample) & (prop_df["Zone"] == "Perivascular")]
    dist_row = prop_df[(prop_df["Sample"] == sample) & (prop_df["Zone"] == "Distal")]
    if peri_row.empty or dist_row.empty:
        continue
    row = {"Sample": sample, "Genotype": geno}
    for ct in typed_ct:
        p = peri_row[ct].values[0] if ct in peri_row.columns else 0
        d = dist_row[ct].values[0] if ct in dist_row.columns else 0
        row[ct] = p / d if d > 0 else np.nan
    enrichment_rows.append(row)

enrich_df = pd.DataFrame(enrichment_rows)
enrich_df["Genotype"] = pd.Categorical(enrich_df["Genotype"], categories=GENO_ORDER, ordered=True)

# Enrichment heatmap per genotype
fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=True)
for ax, geno in zip(axes, GENO_ORDER):
    geno_data = enrich_df[enrich_df["Genotype"] == geno]
    if geno_data.empty:
        ax.set_title(f"{geno} (n=0)")
        continue
    vals = geno_data[typed_ct].median()
    hm = pd.DataFrame({"Enrichment": vals}).T
    sns.heatmap(hm, annot=True, fmt=".2f", cmap="RdBu_r", center=1.0,
                ax=ax, cbar=False, linewidths=0.5, vmin=0, vmax=3)
    ax.set_title(f"{geno} (n={len(geno_data)})")
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right", fontsize=8)

fig.suptitle("Perivascular enrichment (vs Distal) — White Pulp", y=1.02)
plt.tight_layout()
save_figure(fig, "H17_perivascular_enrichment_heatmap")
plt.show()

# --- Strip/box plots: perivascular enrichment per cell type vs genotype ---
# Focus on major cell types with enough variation
major_ct = [ct for ct in typed_ct if ct in enrich_df.columns and enrich_df[ct].notna().sum() >= 3]
n_ct = len(major_ct)
n_cols = min(4, n_ct)
n_rows = (n_ct + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(4 * n_cols, 4 * n_rows), squeeze=False)
axes_flat = axes.flatten()

for i, ct in enumerate(major_ct):
    ax = axes_flat[i]
    plot_data = enrich_df[["Sample", "Genotype", ct]].dropna(subset=[ct])
    sns.boxplot(data=plot_data, x="Genotype", y=ct, order=GENO_ORDER,
                palette=GENO_PALETTE, ax=ax, width=0.5, fliersize=0)
    sns.stripplot(data=plot_data, x="Genotype", y=ct, order=GENO_ORDER,
                  color="black", ax=ax, size=5, alpha=0.7)
    ax.set_title(ct, fontsize=10)
    ax.set_ylabel("Enrichment\n(Perivascular/Distal)")
    ax.axhline(1.0, color="grey", linestyle="--", linewidth=0.8)

    # Dosage annotation
    stats = full_stats_table(plot_data, ct, label=ct)
    _annotate_dosage(ax, stats)

for i in range(len(major_ct), len(axes_flat)):
    axes_flat[i].set_visible(False)

fig.suptitle("Perivascular enrichment by genotype — White Pulp", y=1.02)
plt.tight_layout()
save_figure(fig, "H17_perivascular_enrichment_boxplots")
plt.show()

In [ ]:
# ── Cell 11: Statistical Tests ──

# Per cell type × zone: full_stats_table for genotype comparison
# Focus: perivascular zone in white pulp, PALS, and Follicle

stat_frames = []
for compartment in ["White Pulp", "Follicle", "PALS"]:
    if compartment == "White Pulp":
        sub = cells[cells["in_white_pulp"]]
    elif compartment == "Follicle":
        sub = cells[cells[dist_fol_col] < 0]
    else:
        sub = cells[cells[dist_pals_col] < 0]

    for zone in ["Perivascular", "Proximal", "Distal"]:
        zone_sub = sub[sub["vessel_zone"] == zone]
        if len(zone_sub) == 0:
            continue

        # Per-image cell-type proportions
        zone_prop_rows = []
        for sample in zone_sub["Sample"].unique():
            geno = GENOTYPE_MAP.get(sample)
            if geno is None:
                continue
            s_cells = zone_sub[zone_sub["Sample"] == sample]
            n = len(s_cells)
            if n == 0:
                continue
            ct_counts = s_cells["cell_type"].value_counts()
            row = {"Sample": sample, "Genotype": geno}
            for ct in typed_ct:
                row[ct] = ct_counts.get(ct, 0) / n
            zone_prop_rows.append(row)

        if not zone_prop_rows:
            continue
        zone_prop = pd.DataFrame(zone_prop_rows)
        zone_prop["Genotype"] = pd.Categorical(
            zone_prop["Genotype"], categories=GENO_ORDER, ordered=True)

        for ct in typed_ct:
            if ct not in zone_prop.columns or zone_prop[ct].notna().sum() < 3:
                continue
            label = f"{ct} | {compartment} | {zone}"
            st = full_stats_table(zone_prop, ct, label=label)
            stat_frames.append(st)

if stat_frames:
    all_stats = pd.concat(stat_frames, ignore_index=True)
    save_table(all_stats, "H17_perivascular_stats")
    print(f"Saved {len(all_stats)} statistical test rows")

    # Highlight significant results (p < 0.1)
    sig = all_stats[(all_stats["p"] < 0.1) & (all_stats["Test"] != "Kruskal-Wallis")]
    if len(sig):
        print(f"\nSignificant results (p < 0.1):")
        for _, r in sig.iterrows():
            print(f"  {r['Test']:30s}  {r['Metric']:40s}  p={r['p']:.4f}  {r['Effect_Size']}")
    else:
        print(f"\nNo results with p < 0.1 (expected with n={len(cells['Sample'].unique())} in dev mode)")
else:
    print("No statistical tests could be run (insufficient data)")

# Save cell-type proportions
save_table(prop_df, "H17_cell_type_proportions")
print(f"\nSaved cell-type proportion table: {len(prop_df)} rows")

In [ ]:
# ── Cell 12: Fast-Reload Checkpoint ──
# Save RESTORE-normalized cell data + distances + metadata for rapid gating iteration

checkpoint_path = PROJECT / "analysis" / "H17_restore_cells.parquet"
cells.to_parquet(checkpoint_path, index=False)
print(f"Saved checkpoint: {checkpoint_path}")
print(f"  {len(cells):,} cells, {cells.shape[1]} columns, "
      f"{checkpoint_path.stat().st_size / 1e6:.1f} MB")

---
## Fast-Reload: Resume from Checkpoint
Run Cell 13 below to skip Cells 3-5 (data loading, QC, RESTORE). Then jump to Cell 6 to iterate on gating rules.

In [ ]:
# ── Cell 13: [Fast-Reload] Resume from Checkpoint ──
# Run Cell 1 + Cell 2 first, then this cell, then skip to Cell 6.

checkpoint_path = PROJECT / "analysis" / "H17_restore_cells.parquet"
cells = pd.read_parquet(checkpoint_path)
print(f"Loaded checkpoint: {len(cells):,} cells, {cells.shape[1]} columns")

# Reconstruct all_marker_cols (needed by Cell 6+)
all_marker_cols = [f"Cell: {m}: Mean" for m in ALL_MARKERS]

# Verify marker columns exist
missing = [c for c in all_marker_cols if c not in cells.columns]
if missing:
    print(f"WARNING: missing marker columns: {missing}")
else:
    print(f"All {len(all_marker_cols)} marker columns present")

# Verify metadata columns
for col in ["Sample", "Genotype", "Platform", "Dosage", "Region",
            "cell_type", "vessel_zone", "in_white_pulp", "wp_compartment"]:
    if col in cells.columns:
        print(f"  {col}: OK")
    else:
        print(f"  {col}: MISSING — will need to recompute")

print(f"\nSamples: {sorted(cells['Sample'].unique())}")
print(f"Ready to jump to Cell 6 (scored gating)")

---
## Scale-up
Set `DEV_MODE = False` in Cell 1, then re-run all cells for full 11-sample analysis.

In [ ]:
# ── Cell 14: Scale-up Summary ──
# This cell runs after full pipeline with DEV_MODE = False

n_samples = cells["Sample"].nunique()
n_geno = cells["Genotype"].value_counts()
wp = cells[cells["in_white_pulp"]]
peri_wp = wp[wp["vessel_zone"] == "Perivascular"]

print("=" * 60)
print("H17 Pipeline Summary")
print("=" * 60)
print(f"  Total cells (after QC + crop): {len(cells):,}")
print(f"  Samples: {n_samples}")
print(f"  Genotype distribution:")
for g in GENO_ORDER:
    if g in n_geno.index:
        print(f"    {g}: {n_geno[g]:,}")

print(f"\n  White pulp cells: {len(wp):,}")
print(f"  Perivascular (WP): {len(peri_wp):,}")

print(f"\n  Cell type distribution (all cells):")
for ct in CELL_TYPE_ORDER:
    n = (cells["cell_type"] == ct).sum()
    if n > 0:
        print(f"    {ct:18s} {n:>8,}  ({100*n/len(cells):5.1f}%)")

print(f"\n  Untyped fraction: {100*(cells['cell_type']=='Untyped').mean():.1f}%")

# Platform breakdown if not dev mode
if not DEV_MODE:
    print(f"\n  Platform breakdown:")
    for plat in ["Phenocycler", "CODEX"]:
        plat_cells = cells[cells["Platform"] == plat]
        n_s = plat_cells["Sample"].nunique()
        print(f"    {plat}: {len(plat_cells):,} cells, {n_s} samples")

print("=" * 60)
print(f"\nOutputs:")
print(f"  Figures: H17_marker_heatmap, H17_spatial_celltype_maps,")
print(f"           H17_perivascular_stacked_bar, H17_perivascular_enrichment_heatmap,")
print(f"           H17_perivascular_enrichment_boxplots")
print(f"  Tables:  H17_restore_diagnostics, H17_perivascular_stats,")
print(f"           H17_cell_type_proportions")
print(f"  Checkpoint: H17_restore_cells.parquet")